# Experiment 6 — A Simple Neural Network

**Course:** CSAI4014 — Algorithms for Intelligent Systems and Robotics  
**Experiment:** Two-layer NumPy neural network trained on XOR using backpropagation

Fill in your name, roll number and batch in the lab record before submission.

## Aim and objectives
Implement a two-layer neural network using NumPy alone; show why XOR needs a hidden layer; implement forward pass, squared-error loss, backpropagation and gradient descent; compare learning rates and plot the decision boundary.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

X = np.array([[0,0], [0,1], [1,0], [1,1]], dtype=float)
y = np.array([[0], [1], [1], [0]], dtype=float)

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

print('XOR input pairs:\\n', X)
print('Targets:', y.ravel())

## 1. Forward pass and initial loss
Use four hidden units. Weights are randomly initialised with a fixed seed; biases start at zero.

In [ ]:
rng = np.random.default_rng(1)
hidden = 4
W1 = rng.normal(0, 1, (2, hidden)); b1 = np.zeros((1, hidden))
W2 = rng.normal(0, 1, (hidden, 1)); b2 = np.zeros((1, 1))
h = sigmoid(X @ W1 + b1)
o = sigmoid(h @ W2 + b2)
print('Initial mean squared error:', float(np.mean((o-y)**2)))

## 2. Backpropagation and gradient-descent training

In [ ]:
def train_xor(lr, epochs=20000, seed=1, hidden=4):
    rng = np.random.default_rng(seed)
    W1 = rng.normal(0, 1, (2, hidden)); b1 = np.zeros((1, hidden))
    W2 = rng.normal(0, 1, (hidden, 1)); b2 = np.zeros((1, 1))
    loss = []
    for _ in range(epochs):
        h = sigmoid(X @ W1 + b1)
        o = sigmoid(h @ W2 + b2)
        loss.append(float(np.mean((o-y)**2)))
        do = (o-y) * o * (1-o)
        dW2 = h.T @ do; db2 = do.sum(0, keepdims=True)
        dh = (do @ W2.T) * h * (1-h)
        dW1 = X.T @ dh; db1 = dh.sum(0, keepdims=True)
        W2 -= lr*dW2; b2 -= lr*db2
        W1 -= lr*dW1; b1 -= lr*db1
    return (W1,b1,W2,b2), np.array(loss)

params, loss = train_xor(2.0)
W1,b1,W2,b2 = params
predictions = sigmoid(sigmoid(X @ W1 + b1) @ W2 + b2).ravel()
print('Final loss:', loss[-1])
print('Predictions:', np.round(predictions,3))

## 3. Compare learning rates
Train for 20,000 epochs with learning rates 0.05, 0.5 and 2.0. Use a logarithmic loss axis.

In [ ]:
learning_rates = (0.05, 0.5, 2.0)
runs = {}
plt.figure(figsize=(9,4))
for lr in learning_rates:
    params, loss = train_xor(lr)
    runs[lr] = (params, loss)
    print(f'lr={lr}: final MSE = {loss[-1]:.3e}')
    plt.semilogy(loss, label=f'learning rate = {lr}')
plt.xlabel('epoch'); plt.ylabel('mean squared error'); plt.title('Learning-rate comparison')
plt.legend(); plt.tight_layout(); plt.show()

## 4. Plot the learned decision boundary
Use the trained network at learning rate 2.0 and evaluate it on a dense grid.

In [ ]:
params, loss = runs[2.0]
W1,b1,W2,b2 = params
gx, gy = np.meshgrid(np.linspace(-0.25,1.25,300), np.linspace(-0.25,1.25,300))
G = np.c_[gx.ravel(), gy.ravel()]
Z = sigmoid(sigmoid(G @ W1 + b1) @ W2 + b2).reshape(gx.shape)
plt.figure(figsize=(6,5))
plt.contourf(gx,gy,Z,levels=np.linspace(0,1,11),cmap='Greys')
plt.contour(gx,gy,Z,levels=[0.5],colors='black')
for (x1,x2), target in zip(X,y.ravel()):
    plt.scatter(x1,x2,marker='o' if target==0 else 's',s=90,edgecolors='black',facecolors='white')
plt.xlabel('x1'); plt.ylabel('x2'); plt.title('Learned XOR decision boundary'); plt.tight_layout(); plt.show()
print('Predicted outputs:', np.round(sigmoid(sigmoid(X @ W1 + b1) @ W2 + b2).ravel(),3))

## 5. Try learning rate 10.0
The manual specifically asks you to try 10.0 and record what happens. Run the cell below and describe whether the loss converges, oscillates or diverges on your run.

In [ ]:
params_10, loss_10 = train_xor(10.0)
print('Learning rate 10.0 — final MSE:', loss_10[-1])
print('Finite losses throughout:', bool(np.isfinite(loss_10).all()))
plt.figure(figsize=(8,3)); plt.semilogy(loss_10); plt.xlabel('epoch'); plt.ylabel('MSE'); plt.title('Learning rate = 10.0'); plt.tight_layout(); plt.show()

## Observations and submission checks
Record network shape (2–4–1), seed 1, all learning rates, loss curves, final MSE, four predictions rounded to three decimals, and whether each run converged. State what happened at learning rate 10.0.

**Viva focus:** XOR non-linearity; why zero initialisation fails; sigmoid derivative; backpropagation deltas; relation between gradient descent and hill climbing.